In [2]:
import pandas as pd
import sqlite3

df_customers = pd.DataFrame({
    "customer_id": [1,2,3],
    "name" : ["Jan Kowalski", "Anna Nowak", "Piotr Wiśniewski"],
    "city" : ["Warszawa", "Kraków", "Gdańsk"],
    "age" : [28, 35, 42]
})

young_customers = df_customers[df_customers["age"]<40]
print("Pandas- młodzi klienci")
print(young_customers)

conn = sqlite3.connect(":memory:")
df_customers.to_sql("customers", conn, index=False)

query = """SELECT * FROM customers WHERE age <40"""

sql_result = pd.read_sql_query(query, conn)

print("\nSQL - młodzi klienci:")
print(sql_result)
conn.close()

Pandas- młodzi klienci
   customer_id          name      city  age
0            1  Jan Kowalski  Warszawa   28
1            2    Anna Nowak    Kraków   35

SQL - młodzi klienci:
   customer_id          name      city  age
0            1  Jan Kowalski  Warszawa   28
1            2    Anna Nowak    Kraków   35


In [3]:
%%sql
SELECT * FROM df_customers
WHERE age < 40;

,customer_id,name,city,age
0,1,Jan Kowalski,Warszawa,28
1,2,Anna Nowak,Kraków,35


In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("skle.db") #tworzy baze - plik na dysku
cursor = conn.cursor() # posłaniec do poleceń
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers(
customer_id INTEGER PRIMARY KEY,
name TEXT NOT NULL,
email TEXT UNIQUE,
city TEXT,
registration_date DATE
)
""")
customers_data = [
(1, 'Jan Kowalski', 'jan@email.com', 'Warszawa', '2024-01-10'),
(2, 'Anna Nowak', 'anna@email.com', 'Kraków', '2024-01-12'),
(3, 'Piotr Wiśniewski', 'piotr@email.com', 'Gdańsk', '2024-01-15')
]

cursor.executemany("""
INSERT OR REPLACE INTO customers (customer_id, name, email, city, registration_date)
VALUES(?,?,?,?,?)
""", customers_data)

conn.commit()

df = pd.read_sql_query("SELECT * FROM customers", conn)
print(df)

conn.close()

   customer_id              name            email      city registration_date
0            1      Jan Kowalski    jan@email.com  Warszawa        2024-01-10
1            2        Anna Nowak   anna@email.com    Kraków        2024-01-12
2            3  Piotr Wiśniewski  piotr@email.com    Gdańsk        2024-01-15


In [7]:
%%sql
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT UNIQUE,
    city TEXT,
    registration_date DATE
);

INSERT INTO customers (customer_id, name, email, city, registration_date)
VALUES
    (1, 'Jan Kowalski', 'jan@email.com', 'Warszawa', '2024-01-10'),
    (2, 'Anna Nowak', 'anna@email.com', 'Kraków', '2024-01-12'),
    (3, 'Piotr Wiśniewski', 'piotr@email.com', 'Gdańsk', '2024-01-15')
ON CONFLICT (customer_id) DO UPDATE SET
    name = EXCLUDED.name,
    email = EXCLUDED.email,
    city = EXCLUDED.city,
    registration_date = EXCLUDED.registration_date;

SELECT * FROM customers;

,customer_id,name,email,city,registration_date
0,2,Anna Nowak,anna@email.com,Kraków,2024-01-12
1,1,Jan Kowalski,jan@email.com,Warszawa,2024-01-10
2,3,Piotr Wiśniewski,piotr@email.com,Gdańsk,2024-01-15


In [10]:

import sqlite3
import pandas as pd
conn = sqlite3.connect(':memory:')


customers = pd.DataFrame({
'customer_id': [1, 2, 3, 4],
'name': ['Jan Kowalski', 'Anna Nowak', 'Piotr Wiśniewski', 'Maria Zając'],
'city': ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
})
# Tabela zamówień
orders = pd.DataFrame({
'order_id': [101, 102, 103, 104, 105],
'customer_id': [1, 2, 1, 3, 1],
'order_date': ['2024-01-10', '2024-01-11', '2024-01-12', '2024-01-13',
'2024-01-14'],
'total_amount': [250.0, 120.5, 340.0, 80.0, 190.0]
})
customers.to_sql('customers', conn, index=False)
orders.to_sql('orders', conn, index=False)
# INNER JOIN – tylko klienci którzy złożyli zamówienie
query = """
SELECT
c.customer_id,
c.name,
c.city,
o.order_id,
o.order_date,
o.total_amount
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
ORDER BY c.customer_id, o.order_date
"""
result = pd.read_sql_query(query, conn)
print("INNER JOIN - klienci z zamówieniami:")
print(result)
# Podsumowanie zamówień na klienta
query2 = """
SELECT
c.name,
COUNT(o.order_id) AS number_of_orders,
SUM(o.total_amount) AS total_spent
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name
ORDER BY total_spent DESC
"""
print("\nPodsumowanie zamówień:")
print(pd.read_sql_query(query2, conn))

conn.close()

INNER JOIN - klienci z zamówieniami:
   customer_id              name      city  order_id  order_date  total_amount
0            1      Jan Kowalski  Warszawa       101  2024-01-10         250.0
1            1      Jan Kowalski  Warszawa       103  2024-01-12         340.0
2            1      Jan Kowalski  Warszawa       105  2024-01-14         190.0
3            2        Anna Nowak    Kraków       102  2024-01-11         120.5
4            3  Piotr Wiśniewski    Gdańsk       104  2024-01-13          80.0

Podsumowanie zamówień:
               name  number_of_orders  total_spent
0      Jan Kowalski                 3        780.0
1        Anna Nowak                 1        120.5
2  Piotr Wiśniewski                 1         80.0


Zadanie 1 – Podstawowy SELECT
Utwórz bazę danych SQLite z tabelą products zawierającą kolumny: product_id, name,
category, price, stock. Wstaw 10 przykładowych produktów.
Wymagania:
Napisz zapytanie SELECT wybierające wszystkie produkty
Napisz zapytanie wybierające tylko name i price
Wyświetl wyniki za pomocą pd.read_sql_query() oraz Zadanie 2 – WHERE z warunkami
Używając tabeli products z zadania 1:
Wymagania:
Znajdź produkty droższe niż 100 PLN
Znajdź produkty z kategorii "Elektronika" LUB z ceną < 50 PLN
Znajdź produkty z zapasem (stock) między 10 a 50

In [16]:
import pandas as pd
import _sqlite3

conn = sqlite3.connect(":memory:")

products = pd.DataFrame({
    "product_id" : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "name" :  ["Łosoś", "Farry", "Kurtka", "HarryPotter", "Macbook", "Ostrygi", "Amica", "Sweter","Zbiór zadań", "Iphone17"],
    "category" : 2*["Jedzenie", "Chemia", "Ubrania", "Książki", "Elektronika"],
    "price" : [250, 300, 560, 340, 647, 937, 387, 378, 873, 347],
    "stock" : [ 9, 4, 6, 23, 4, 1, 7, 5, 8, 21]
})
products.to_sql("products", conn, index=False)

query = """
SELECT * FROM products
"""
query2= """
SELECT name, price FROM products
"""

query3 = """
SELECT * FROM products
WHERE price >100
"""

query4 = """
SELECT * FROM products
WHERE category == "Elektronika" OR price<300
"""
query5 ="""
SELECT * FROM products
WHERE stock BETWEEN 10 AND 50
"""

print(f"Wszystko:")
print(pd.read_sql_query(query, conn))
print(f"nazwa i cena: ")
print(pd.read_sql_query(query2, conn))

print(f"Cena większa niż 100:")
print(pd.read_sql_query(query3, conn))
print(f"kategoria elektronika lub cen <300")
print(pd.read_sql_query(query4, conn))
print(f"Stock pomiedzy 10 a 50")
print(pd.read_sql_query(query5, conn))

conn.close()

Wszystko:
   product_id         name     category  price  stock
0           1        Łosoś     Jedzenie    250      9
1           2        Farry       Chemia    300      4
2           3       Kurtka      Ubrania    560      6
3           4  HarryPotter      Książki    340     23
4           5      Macbook  Elektronika    647      4
5           6      Ostrygi     Jedzenie    937      1
6           7        Amica       Chemia    387      7
7           8       Sweter      Ubrania    378      5
8           9  Zbiór zadań      Książki    873      8
9          10     Iphone17  Elektronika    347     21
nazwa i cena: 
          name  price
0        Łosoś    250
1        Farry    300
2       Kurtka    560
3  HarryPotter    340
4      Macbook    647
5      Ostrygi    937
6        Amica    387
7       Sweter    378
8  Zbiór zadań    873
9     Iphone17    347
Cena większa niż 100:
   product_id         name     category  price  stock
0           1        Łosoś     Jedzenie    250      9
1        

In [19]:
%%sql

SELECT * FROM products;



,product_id,name,category,price,stock
0,1,Łosoś,Jedzenie,250,9
1,2,Farry,Chemia,300,4
2,3,Kurtka,Ubrania,560,6
3,4,HarryPotter,Książki,340,23
4,5,Macbook,Elektronika,647,4
5,6,Ostrygi,Jedzenie,937,1
6,7,Amica,Chemia,387,7
7,8,Sweter,Ubrania,378,5
8,9,Zbiór zadań,Książki,873,8
9,10,Iphone17,Elektronika,347,21


In [20]:
%%sql

SELECT name, price FROM products;


,name,price
0,Łosoś,250
1,Farry,300
2,Kurtka,560
3,HarryPotter,340
4,Macbook,647
5,Ostrygi,937
6,Amica,387
7,Sweter,378
8,Zbiór zadań,873
9,Iphone17,347


In [21]:
%%sql

SELECT * FROM products
WHERE price >100;


,product_id,name,category,price,stock
0,1,Łosoś,Jedzenie,250,9
1,2,Farry,Chemia,300,4
2,3,Kurtka,Ubrania,560,6
3,4,HarryPotter,Książki,340,23
4,5,Macbook,Elektronika,647,4
5,6,Ostrygi,Jedzenie,937,1
6,7,Amica,Chemia,387,7
7,8,Sweter,Ubrania,378,5
8,9,Zbiór zadań,Książki,873,8
9,10,Iphone17,Elektronika,347,21


In [25]:
%%sql

SELECT * FROM products
WHERE category == 'Elektronika' OR price<300;



,product_id,name,category,price,stock
0,1,Łosoś,Jedzenie,250,9
1,5,Macbook,Elektronika,647,4
2,10,Iphone17,Elektronika,347,21


 Zadanie 10 – HAVING i grupowanie
Wymagania:
Używając danych sprzedażowych, pogrupuj według kategorii
Użyj HAVING aby znaleźć tylko kategorie z przychodem > 1000 PLN
Oblicz liczbę transakcji, łączny przychód i średnią dla każdej kategorii
Posortuj według łącznego przychodu

In [30]:
import pandas as pd
import _sqlite3

conn = sqlite3.connect(":memory:")


sales = pd.DataFrame({
'sale_id': range(1, 21),
'product': ['Laptop', 'Mysz', 'Klawiatura', 'Monitor', 'Słuchawki'] * 4,
'category': (['Elektronika'] * 2 + ['Akcesoria'] * 3) * 4,
'revenue':
[3500, 50, 200, 1200, 150,
3600, 45, 210, 1250, 140,
3400, 55, 190, 1180, 160,
3550, 48, 205, 1220, 155]
})

sales.to_sql("sales", conn, index=False)

query1 = """
WITH sprzedaz AS(
SELECT *,
COUNT(sale_id) AS liczba_transakcji,
SUM(revenue) AS laczny_przychod,
AVG(revenue) AS sredni_przychod
FROM sales
GROUP BY category
    )
SELECT * FROM sprzedaz
WHERE laczny_przychod >1000
ORDER BY laczny_przychod DESC

"""

print(pd.read_sql_query(query1, conn))

conn.close()



   sale_id     product     category  revenue  liczba_transakcji  \
0        1      Laptop  Elektronika     3500                  8   
1        3  Klawiatura    Akcesoria      200                 12   

   laczny_przychod  sredni_przychod  
0            14248      1781.000000  
1             6260       521.666667  


In [31]:
%%sql
SELECT
category,
COUNT(sale_id) AS liczba_transakcji,
SUM(revenue) AS laczny_przychod,
AVG(revenue) AS sredni_przychod
FROM sales
GROUP BY category
HAVING SUM(revenue) >1000
ORDER BY laczny_przychod DESC;

,category,liczba_transakcji,laczny_przychod,sredni_przychod
0,Elektronika,8,14248.0,1781.000000
1,Akcesoria,12,6260.0,521.666667


Dataset: Titanic
Wymagania:
Znajdź pasażerów 1 klasy, kobiety, które przeżyły
Oblicz średni wiek i średnią cenę biletu dla tej grupy
Porównaj z mężczyznami z 3 klasy którzy nie przeżyli
Stwórz wykres porównawczy (barplot)

In [1]:
%%sql
SELECT
'Kobiety (1 klasa, przeżyły)' AS grupa,
AVG(Age) AS sredni_wiek,
AVG(Fare) AS srednia_cena
FROM titanic
WHERE Pclass = 1 AND Sex = 'female' AND Survived = 1

UNION ALL

SELECT
'Mężczyźni (3 klasa, utoneli)' AS grupa,
AVG(Age) AS sredni_wiek,
AVG(Fare) AS srednia_cena
FROM titanic
WHERE Pclass = 3 AND Sex = 'male' AND Survived = 0;

,grupa,sredni_wiek,srednia_cena
0,"Kobiety (1 klasa, przeżyły)",34.939024,105.978159
1,"Mężczyźni (3 klasa, utoneli)",27.255814,12.204469
